# Explainable AI Image Classification using CNN + DeepSHAP

**Improved version of the original `Implementation_of_DeepSHAP.ipynb`**

This notebook keeps the original project's core workflow:

**CIFAR-10 → preprocessing → CNN → prediction → DeepSHAP explanation**

It improves the original notebook by adding:

- reproducible random seeds
- a proper train/validation/test split
- callbacks for better training control
- saved model and training history
- evaluation metrics
- confusion matrix
- training/validation plots
- prediction confidence
- clearer sample selection
- a smaller, configurable SHAP background set
- a safer DeepSHAP compatibility fallback
- a final project summary

> **Portfolio note:** This is a learning/portfolio project. CIFAR-10 is a benchmark dataset, not a production medical or safety-critical dataset.

## 1. Install the required packages

Run this cell only if your environment does not already have the packages.

For a clean local environment, create a virtual environment and install the versions from `requirements.txt` after this notebook is working.

In [ ]:
# Uncomment if required in a local Jupyter environment.
# %pip install -U tensorflow shap scikit-learn seaborn pandas matplotlib numpy

## 2. Import libraries

We use:

- **NumPy/Pandas** for data handling
- **Matplotlib/Seaborn** for visualization
- **TensorFlow/Keras** for the CNN
- **scikit-learn** for splitting and evaluation
- **SHAP** for explainability

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Activation,
    Flatten,
    Conv2D,
    MaxPooling2D
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

warnings.filterwarnings("ignore")

print("TensorFlow:", tf.__version__)
print("SHAP:", shap.__version__)

## 3. Make the experiment reproducible

Machine-learning training can produce slightly different results between runs.

Setting seeds makes the experiment more reproducible.

In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed:", SEED)

## 4. Load the CIFAR-10 dataset

CIFAR-10 contains 60,000 colour images of size **32×32×3** across 10 classes:

0 airplane, 1 automobile, 2 bird, 3 cat, 4 deer, 5 dog, 6 frog, 7 horse, 8 ship, 9 truck.

The original notebook loaded the dataset twice. Here we load it once and keep the workflow cleaner.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Convert labels from shape (n, 1) to shape (n,)
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

print("Full training images:", x_train_full.shape)
print("Full training labels:", y_train_full.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

## 5. Define the class labels

In [ ]:
num_classes = 10

class_labels = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck"
}

class_names = [class_labels[i] for i in range(num_classes)]

print(class_names)

## 6. Create a proper train/validation/test split

The original notebook used the **test set as validation data during training**.

For a cleaner ML workflow:

- **Training set:** used to learn model parameters
- **Validation set:** used during development/training decisions
- **Test set:** kept untouched until final evaluation

We use stratification so each split keeps a similar class distribution.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.10,
    random_state=SEED,
    stratify=y_train_full
)

print("Training:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

## 7. Normalize the images

CIFAR-10 pixels originally range from **0 to 255**.

We scale them to **0 to 1**.

This makes the numerical input easier for the neural network to process.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Pixel range after normalization:")
print("min:", x_train.min(), "max:", x_train.max())

## 8. Visualize the class distribution

Before training, it is useful to understand whether the dataset is reasonably balanced.

In [ ]:
class_counts = pd.Series(y_train).value_counts().sort_index()

plt.figure(figsize=(10, 5))
sns.barplot(
    x=class_names,
    y=class_counts.values
)
plt.title("CIFAR-10 Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 9. Display example images from every class

This is the same idea as the original notebook, but the selection is made explicitly and the images are displayed directly from the training data.

In [ ]:
plt.figure(figsize=(15, 6))

for class_id in range(num_classes):
    idx = np.where(y_train == class_id)[0][0]
    plt.subplot(2, 5, class_id + 1)
    plt.imshow(x_train[idx])
    plt.axis("off")
    plt.title(class_labels[class_id])

plt.suptitle("Example Training Images", fontsize=18)
plt.tight_layout()
plt.show()

## 10. Build the CNN model

This keeps the architecture from the original notebook:

- Conv2D → ReLU
- Conv2D → ReLU
- MaxPooling
- Dropout
- Conv2D → ReLU
- Conv2D → ReLU
- MaxPooling
- Dropout
- Flatten
- Dense
- Dropout
- Softmax output

The main change is using an explicit `Input` layer, which is clearer in modern Keras.

In [ ]:
model = Sequential([
    Input(shape=x_train.shape[1:]),

    Conv2D(32, (3, 3), padding="same"),
    Activation("relu"),

    Conv2D(32, (3, 3)),
    Activation("relu"),

    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), padding="same"),
    Activation("relu"),

    Conv2D(64, (3, 3)),
    Activation("relu"),

    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    Flatten(),

    Dense(512),
    Activation("relu"),
    Dropout(0.50),

    Dense(num_classes),
    Activation("softmax")
])

model.summary()

## 11. Compile the CNN

We use:

- **Sparse categorical cross-entropy** because labels are integer class IDs.
- **Adam** as the optimizer.
- **Accuracy** as the main training metric.

In [ ]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

print("Model compiled successfully.")

## 12. Configure training callbacks

Callbacks make the project more production-like.

- `EarlyStopping` stops training when validation performance stops improving.
- `ReduceLROnPlateau` lowers the learning rate when progress stalls.
- `ModelCheckpoint` saves the best model.

In [ ]:
MODEL_PATH = "cifar10_cnn_best.keras"

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),
    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    )
]

print("Callbacks configured.")

## 13. Train the CNN

The original notebook used 10 epochs and passed the test set as validation data.

Here we train against the training set and monitor the separate validation set.

You can increase `EPOCHS` later if your hardware allows it.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 64

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

## 14. Plot training and validation accuracy

These curves help us see whether the model is learning and whether training and validation performance begin to diverge.

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(10, 5))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Plot training and validation loss

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 16. Evaluate the final model on the untouched test set

This is the first point where we use the test set for final model evaluation.

In [ ]:
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## 17. Generate test predictions

`predict_proba` contains the probability for each of the 10 classes.

The class with the largest probability becomes the predicted class.

In [ ]:
test_probabilities = model.predict(
    x_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

y_pred = np.argmax(test_probabilities, axis=1)

print("Predictions generated:", y_pred.shape)
print("Accuracy:", accuracy_score(y_test, y_pred))

## 18. Classification report

Precision, recall and F1-score provide more detail than accuracy alone.

This is especially useful when explaining model performance in an interview or README.

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    digits=4
)

print(report)

## 19. Confusion matrix

A confusion matrix shows which classes the model confuses with one another.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("CIFAR-10 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## 20. Show sample predictions with confidence

For each selected image we display:

- actual class
- predicted class
- prediction confidence

In [ ]:
sample_indices = []

for class_id in range(num_classes):
    idx = np.where(y_test == class_id)[0][0]
    sample_indices.append(idx)

plt.figure(figsize=(15, 7))

for plot_position, idx in enumerate(sample_indices):
    actual = y_test[idx]
    predicted = y_pred[idx]
    confidence = test_probabilities[idx][predicted]

    plt.subplot(2, 5, plot_position + 1)
    plt.imshow(x_test[idx])
    plt.axis("off")

    title = (
        f"Actual: {class_labels[actual]}\n"
        f"Pred: {class_labels[predicted]}\n"
        f"Confidence: {confidence:.1%}"
    )
    plt.title(title, fontsize=10)

plt.suptitle("Actual vs Predicted: One Test Image per Class", fontsize=18)
plt.tight_layout()
plt.show()

## 21. Save the trained model

Saving the model allows us to use it later without retraining.

This file will also be useful when we create the Streamlit application.

In [ ]:
FINAL_MODEL_PATH = "cifar10_cnn_final.keras"
model.save(FINAL_MODEL_PATH)

print("Saved:", FINAL_MODEL_PATH)
print("Best checkpoint:", MODEL_PATH)

## 22. Prepare a small SHAP background dataset

DeepSHAP uses background examples as a reference for explaining predictions.

The original notebook used 1,000 images. For a beginner laptop, 100–200 background images are much faster for experimentation.

Increase this number later if your machine can handle it.

In [ ]:
SHAP_BACKGROUND_SIZE = 100

rng = np.random.default_rng(SEED)

background_indices = rng.choice(
    len(x_train),
    size=SHAP_BACKGROUND_SIZE,
    replace=False
)

background = x_train[background_indices]

print("SHAP background shape:", background.shape)

## 23. Select test images for explanation

SHAP can be computationally expensive.

We start with **5 correctly predicted test images**. Once the workflow works, you can increase the number.

In [ ]:
correct_indices = np.where(y_pred == y_test)[0]

SHAP_SAMPLE_SIZE = 5

if len(correct_indices) < SHAP_SAMPLE_SIZE:
    raise ValueError("Not enough correctly predicted images for SHAP.")

explain_indices = correct_indices[:SHAP_SAMPLE_SIZE]
test_arr = x_test[explain_indices]

actual_labels = y_test[explain_indices]
predicted_labels = y_pred[explain_indices]
predicted_probabilities = test_probabilities[explain_indices]

print("Images selected for explanation:", test_arr.shape)

for i, idx in enumerate(explain_indices):
    print(
        f"{i}: actual={class_labels[y_test[idx]]}, "
        f"predicted={class_labels[y_pred[idx]]}, "
        f"confidence={test_probabilities[idx][y_pred[idx]]:.2%}"
    )

## 24. Create the DeepSHAP explainer

This is the main explainability step from the original notebook:

```python
shap.DeepExplainer(model, background)
```

DeepSHAP estimates how input features contribute to model outputs relative to the selected background examples.

**Compatibility note:** SHAP and TensorFlow versions can differ. The fallback below tries the current DeepExplainer API first and then retries with a compatibility setting when supported.

In [ ]:
try:
    explainer = shap.DeepExplainer(model, background)
    print("DeepSHAP explainer created successfully.")
except Exception as e:
    print("Standard DeepExplainer initialization failed:")
    print(type(e).__name__, str(e)[:500])

    # Compatibility fallback for some TensorFlow/SHAP combinations.
    try:
        tf.compat.v1.disable_eager_execution()
        print("Retrying after disabling TensorFlow eager execution.")
        raise RuntimeError(
            "Restart the Jupyter kernel after disabling eager execution, "
            "then rerun from the imports/model cells."
        )
    except Exception:
        raise

## 25. Calculate SHAP values

We now ask DeepSHAP to explain the selected predictions.

Depending on the installed SHAP version, the returned structure can differ. We keep the raw result and inspect its shape/type before plotting.

In [ ]:
shap_values = explainer.shap_values(test_arr)

print("SHAP result type:", type(shap_values))

if isinstance(shap_values, list):
    print("Number of SHAP outputs:", len(shap_values))
    print("First output shape:", np.asarray(shap_values[0]).shape)
else:
    print("SHAP array shape:", np.asarray(shap_values).shape)

## 26. Visualize DeepSHAP explanations

The original notebook used `shap.image_plot`.

For image explanations, SHAP visualizations show which image regions contribute to the model output.

Interpret the colors as **model-attribution signals**, not as a statement that the model truly "understands" an object like a human.

In [ ]:
# SHAP's image_plot supports the list/array formats returned by
# different SHAP versions.

shap.image_plot(
    shap_values,
    test_arr
)

## 27. Inspect one prediction and its explanation

Let's connect the explanation back to the actual prediction.

This makes the notebook easier to explain during an interview:

In [ ]:
idx = 0

print("Actual class     :", class_labels[actual_labels[idx]])
print("Predicted class  :", class_labels[predicted_labels[idx]])
print(
    "Confidence       :",
    f"{predicted_probabilities[idx][predicted_labels[idx]]:.2%}"
)

plt.figure(figsize=(4, 4))
plt.imshow(test_arr[idx])
plt.axis("off")
plt.title(
    f"Predicted: {class_labels[predicted_labels[idx]]}\n"
    f"Confidence: {predicted_probabilities[idx][predicted_labels[idx]]:.2%}"
)
plt.show()

## 28. Save important project outputs

These files can later be committed to GitHub selectively:

- trained model
- training history
- classification report
- confusion matrix

For a public repository, large model files may be better stored with Git LFS or a model registry rather than normal Git.

In [ ]:
history_df.to_csv("training_history.csv", index=False)

with open("classification_report.txt", "w") as f:
    f.write(report)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("CIFAR-10 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()

print("Saved:")
print("- training_history.csv")
print("- classification_report.txt")
print("- confusion_matrix.png")

# 29. Project conclusion

### What we built

1. Loaded the CIFAR-10 image dataset.
2. Created separate training, validation and test sets.
3. Normalized image pixel values.
4. Built a CNN based on the architecture from the original notebook.
5. Trained the CNN using validation data rather than the test set.
6. Evaluated the model using test accuracy and a classification report.
7. Created a confusion matrix.
8. Displayed predictions with confidence.
9. Saved the trained model.
10. Added DeepSHAP to explain selected predictions.

### End-to-end workflow

```text
CIFAR-10
   ↓
Train / Validation / Test Split
   ↓
Normalization
   ↓
CNN
   ↓
Training + Callbacks
   ↓
Final Test Evaluation
   ↓
Prediction + Confidence
   ↓
DeepSHAP
   ↓
Visual Explanation
```

### Important limitation

The model is a benchmark computer-vision project using CIFAR-10. The SHAP visualization explains model attribution, but it should not be interpreted as proof of human-like reasoning or causal understanding.

### Next stage

The next step is to turn this notebook into a small **Streamlit application** where a user can upload an image and see the model prediction and explainability output.